In [ ]:
!pip install evaluate
!pip install bert_score

In [ ]:
import pandas as pd
from evaluate import load
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq


##EXERCISE 1

In [ ]:
def create_transformers_train_data(sentences, translations, tokenizer):
    inputs_en = tokenizer(sentences, max_length=10, truncation=True)

    with tokenizer.as_target_tokenizer():
        outputs_es = tokenizer(translations, max_length=10, truncation=True)

    data = Dataset.from_dict({'input_ids': inputs_en['input_ids'],
                              'attention_mask': inputs_en['attention_mask'],
                              'labels': outputs_es['input_ids']})
    return data



def train_transformer(model, train_loader, optimizer, epochs=5, device='cpu'):
    model = model.to(device)
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(train_loader))
        print(f'Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/NLP2025/yelp_parallel/yelp_parallel/test_en_parallel.txt', sep='\t', header=None)
data.head()

In [ ]:
from transformers import T5Tokenizer
from torch.utils.data import DataLoader

model_name = 't5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
train_dataset = create_transformers_train_data(
    data[0].values.tolist(),
    data[1].values.tolist(),
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)


Train 't5-small'

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=data_collator)

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

optimizer = AdamW(model.parameters(), lr=0.001)

train_transformer(model, train_loader, optimizer, 5)

In [ ]:
def decode_with_transformer(sentence, tokenizer, model, device='cpu'):
    model = model.to(device)
    model.eval()
    tokens = tokenizer([sentence], return_tensors='pt').to(device)
    out = model.generate(**tokens, max_length=10)

    with tokenizer.as_target_tokenizer():
        pred_sentence = tokenizer.decode(out[0], skip_special_tokens=True)

    return pred_sentence

Evaluation with T5

In [ ]:
predictions = []
references = []

for i in range(len(data)):
    sentence = data[0].iloc[i]
    pred = decode_with_transformer(sentence, tokenizer, model)
    ref = data[1].iloc[i]

    predictions.append(pred)
    references.append([ref])

In [ ]:
bleu = load('bleu')

In [ ]:
bleu_results = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU score:", bleu_results["bleu"])

In [ ]:
bertscore = load('bertscore')

In [ ]:
from bert_score import score

P, R, F1 = score(
    predictions,
    [ref[0] for ref in references],
    lang="en",
    model_type="bert-base-uncased"
)

In [ ]:
print("\nPrecision:", P.mean().item())
print("Recall:", R.mean().item())
print("F1:", F1.mean().item())

Train 'FLAN-T5'

In [ ]:
model_name = 'google/flan-t5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
train_dataset = create_transformers_train_data(
    data[0].values.tolist(),
    data[1].values.tolist(),
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=data_collator)

In [ ]:
optimizer = AdamW(model.parameters(), lr=0.001)

train_transformer(model, train_loader, optimizer, 5)

Evaluation with FLAN-T5

In [ ]:
predictions = []
references = []

for i in range(len(data)):
    sentence = data[0].iloc[i]
    pred = decode_with_transformer(sentence, tokenizer, model)
    ref = data[1].iloc[i]

    predictions.append(pred)
    references.append([ref])

In [ ]:
bleu_results = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU score:", bleu_results["bleu"])

P, R, F1 = score(
    predictions,
    [ref[0] for ref in references],
    lang="en",
    model_type="bert-base-uncased"
)


print("\nPrecision:", P.mean().item())
print("Recall:", R.mean().item())
print("F1:", F1.mean().item())


##EXERCISE 2

In [ ]:
model_name = 't5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
instruction = 'Transform from Negative to Positive: '

In [ ]:
negative_sentences_with_ins = [f'{instruction}{s}' for s in data[0].values.tolist()]
negative_sentences_with_ins

In [ ]:
train_dataset = create_transformers_train_data(
    negative_sentences_with_ins,
    data[1].values.tolist(),
    tokenizer
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=data_collator)

In [ ]:
optimizer = AdamW(model.parameters(), lr=0.001)

train_transformer(model, train_loader, optimizer, 5)

Evaluation with instructions

In [ ]:
predictions = []
references = []

for i in range(len(data)):
    sentence = data[0].iloc[i]
    pred = decode_with_transformer(sentence, tokenizer, model)
    ref = data[1].iloc[i]

    predictions.append(pred)
    references.append([ref])

In [ ]:
bleu_results = bleu.compute(
    predictions=predictions,
    references=references
)

print("BLEU score:", bleu_results["bleu"])

P, R, F1 = score(
    predictions,
    [ref[0] for ref in references],
    lang="en",
    model_type="bert-base-uncased"
)


print("\nPrecision:", P.mean().item())
print("Recall:", R.mean().item())
print("F1:", F1.mean().item())


Conclusion:

When the T5 model was trained without instructions, both BLEU and BERTScore were higher (BLEU ≈ 0.29, F1 ≈ 0.70), indicating that the model more closely reproduced the reference sentences in terms of words and structure.

With instructional fine-tuning, BLEU and BERTScore dropped significantly (BLEU ≈ 0.11, F1 ≈ 0.56). This is because the model now attempts to follow an abstract instruction (“transform a negative sentence into a positive one”), which allows for more diverse expressions and less direct overlap with the reference.

##EXERCISE 3

Tрансформација на реченици кои содржат негативен сентимент во реченици
кои содржат позитивен сентимент. (the same as the exercise 2)